In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_file = project_root / "data" / "raw" / "retail_sales_dataset.xlsx"

source_tables = {
    "Customers": pd.read_excel(data_file, sheet_name="Customers"),
    "Products": pd.read_excel(data_file, sheet_name="Products"),
    "Stores": pd.read_excel(data_file, sheet_name="Stores"),
    "Transactions": pd.read_excel(data_file, sheet_name="Transactions")
}

customers = source_tables["Customers"].copy()
products = source_tables["Products"].copy()
stores = source_tables["Stores"].copy()
transactions = source_tables["Transactions"].copy()

customers["BirthDate"] = pd.to_datetime(customers["BirthDate"], errors="coerce")
customers["JoinDate"] = pd.to_datetime(customers["JoinDate"], errors="coerce")
transactions["Date"] = pd.to_datetime(transactions["Date"], errors="coerce")

analytical_df = transactions.merge(customers, on="CustomerID", how="left")
analytical_df = analytical_df.merge(products, on="ProductID", how="left")
analytical_df = analytical_df.merge(
    stores,
    on="StoreID",
    how="left",
    suffixes=("_Customer", "_Store")
)

analytical_df = analytical_df.rename(
    columns={
        "Date": "TransactionDate",
        "City_Customer": "CustomerCity",
        "City_Store": "StoreCity",
        "Region": "StoreRegion"
    }
)

analytical_df["gross_sales"] = analytical_df["Quantity"] * analytical_df["UnitPrice"]
analytical_df["discount_amount"] = analytical_df["gross_sales"] * analytical_df["Discount"]
analytical_df["net_sales"] = analytical_df["gross_sales"] - analytical_df["discount_amount"]
analytical_df["total_cost"] = analytical_df["Quantity"] * analytical_df["CostPrice"]
analytical_df["profit"] = analytical_df["net_sales"] - analytical_df["total_cost"]

analytical_df["year"] = analytical_df["TransactionDate"].dt.year
analytical_df["month"] = analytical_df["TransactionDate"].dt.month
analytical_df["year_month"] = analytical_df["TransactionDate"].dt.to_period("M").astype(str)

analytical_df.head()

,TransactionID,TransactionDate,CustomerID,ProductID,StoreID,Quantity,Discount,PaymentMethod,FirstName,LastName,...,StoreCity,StoreRegion,gross_sales,discount_amount,net_sales,total_cost,profit,year,month,year_month
0,T00001,2024-06-18,C160,P014,S003,1,0.10,Bank Transfer,Meagan,Macdonald,...,New Michele,West,1342.75,134.275,1208.475,797.94,410.535,2024,6,2024-06
1,T00002,2023-11-02,C171,P030,S004,3,0.15,Bank Transfer,Christina,Dominguez,...,Brianahaven,North,87.72,13.158,74.562,45.84,28.722,2023,11,2023-11
2,T00003,2024-03-28,C142,P002,S002,2,0.15,Mobile Money,Suzanne,Fox,...,Peckmouth,East,1637.52,245.628,1391.892,1055.24,336.652,2024,3,2024-03
3,T00004,2024-06-15,C174,P050,S002,5,0.10,Mobile Money,Scott,Howell,...,Peckmouth,East,5223.20,522.320,4700.880,3875.35,825.530,2024,6,2024-06
4,T00005,2024-08-29,C141,P036,S001,3,0.10,Credit Card,Adam,Lucas,...,Jimenezborough,South,4504.38,450.438,4053.942,3503.19,550.752,2024,8,2024-08


In [2]:
quality_summary = pd.DataFrame({
    "columna": analytical_df.columns,
    "tipo_dato": analytical_df.dtypes.astype(str).values,
    "nulos": analytical_df.isna().sum().values,
    "porcentaje_nulos": (analytical_df.isna().mean() * 100).round(2).values,
    "valores_unicos": analytical_df.nunique(dropna=False).values
})

quality_summary.sort_values("porcentaje_nulos", ascending=False).head(20)

,columna,tipo_dato,nulos,porcentaje_nulos,valores_unicos
0,TransactionID,str,0,0.0,5000
1,TransactionDate,datetime64[us],0,0.0,729
2,CustomerID,str,0,0.0,200
3,ProductID,str,0,0.0,50
4,StoreID,str,0,0.0,5
5,Quantity,int64,0,0.0,5
6,Discount,float64,0,0.0,4
7,PaymentMethod,str,0,0.0,4
8,FirstName,str,0,0.0,137
9,LastName,str,0,0.0,161


In [3]:
duplicate_transactions = analytical_df["TransactionID"].duplicated().sum()

print(f"Transacciones duplicadas por TransactionID: {duplicate_transactions}")
print(f"Filas totales: {analytical_df.shape[0]:,}")
print(f"Columnas totales: {analytical_df.shape[1]:,}")
print(f"Rango de fechas: {analytical_df['TransactionDate'].min()} a {analytical_df['TransactionDate'].max()}")

Transacciones duplicadas por TransactionID: 0
Filas totales: 5,000
Columnas totales: 30
Rango de fechas: 2023-09-10 00:00:00 a 2025-09-09 00:00:00


In [4]:
executive_metrics = {
    "transactions": analytical_df["TransactionID"].nunique(),
    "net_sales": analytical_df["net_sales"].sum(),
    "profit": analytical_df["profit"].sum(),
    "average_ticket": analytical_df["net_sales"].mean()
}

monthly_summary = (
    analytical_df.groupby("year_month", as_index=False)
    .agg(
        net_sales=("net_sales", "sum"),
        profit=("profit", "sum"),
        transactions=("TransactionID", "nunique")
    )
    .sort_values("year_month")
)

executive_metrics, monthly_summary.tail()

({'transactions': 5000,
  'net_sales': np.float64(14301903.147499999),
  'profit': np.float64(3826314.6674999995),
  'average_ticket': np.float64(2860.3806295)},
    year_month    net_sales       profit  transactions
 20    2025-05  632784.8700  170728.1600           221
 21    2025-06  679817.8090  181803.2990           224
 22    2025-07  610693.2775  157140.1975           214
 23    2025-08  587098.4135  150720.7635           204
 24    2025-09  145960.4105   39771.8605            49)

In [5]:
sales_by_category = (
    analytical_df.groupby("Category", as_index=False)["net_sales"]
    .sum()
    .sort_values("net_sales", ascending=False)
)

profit_by_region = (
    analytical_df.groupby("StoreRegion", as_index=False)["profit"]
    .sum()
    .sort_values("profit", ascending=False)
)

top_products = (
    analytical_df.groupby("ProductName", as_index=False)
    .agg(
        net_sales=("net_sales", "sum"),
        profit=("profit", "sum"),
        quantity=("Quantity", "sum")
    )
    .sort_values("net_sales", ascending=False)
    .head(10)
)

top_subcategories = (
    analytical_df.groupby("SubCategory", as_index=False)
    .agg(
        net_sales=("net_sales", "sum"),
        profit=("profit", "sum"),
        quantity=("Quantity", "sum")
    )
    .sort_values("net_sales", ascending=False)
    .head(10)
)

sales_by_payment = (
    analytical_df.groupby("PaymentMethod", as_index=False)["net_sales"]
    .sum()
    .sort_values("net_sales", ascending=False)
)

sales_by_category.head(), top_products.head()

(      Category     net_sales
 0  Electronics  6.316917e+06
 1      Fashion  6.232280e+06
 2    Groceries  1.752706e+06,
            ProductName    net_sales       profit  quantity
 6      Book Television  605028.6025  115950.6125       337
 2         And Footwear  585613.0225  244603.2625       336
 40           Set Dairy  573777.0780  242272.2780       330
 45  Traditional Laptop  521031.0060   94196.8460       317
 4     Beat Accessories  498224.7415  100808.8915       299)

In [6]:
customer_label = "CustomerName" if "CustomerName" in analytical_df.columns else "CustomerID"

customer_summary = (
    analytical_df.groupby(["CustomerID", customer_label], as_index=False)
    .agg(
        transactions=("TransactionID", "nunique"),
        net_sales=("net_sales", "sum"),
        profit=("profit", "sum"),
        quantity=("Quantity", "sum")
    )
    .sort_values("net_sales", ascending=False)
)

customer_summary.head(10)

,CustomerID,transactions,net_sales,profit,quantity
11,C012,32,117085.7640,28427.1340,100
109,C110,31,112255.4275,29215.4975,103
84,C085,34,111867.5625,30495.9825,105
76,C077,32,102378.9335,24664.7535,83
41,C042,26,102068.0475,35225.7275,86
167,C168,28,101478.7265,26138.9465,91
185,C186,36,101239.3985,25368.1585,112
28,C029,30,100719.7115,29024.5115,95
101,C102,34,99498.9450,26013.9750,99
32,C033,28,97397.9355,27254.8455,81


In [7]:
numeric_summary = analytical_df.select_dtypes(include="number").describe().T
numeric_summary

,count,mean,std,min,25%,50%,75%,max
Quantity,5000.0,2.989800,1.409430,1.0000,2.000000,3.000,4.00000,5.0000
Discount,5000.0,0.075880,0.056497,0.0000,0.050000,0.100,0.15000,0.1500
UnitPrice,5000.0,1036.606538,589.316649,25.5700,428.050000,1193.090,1487.41000,1952.6500
CostPrice,5000.0,701.998512,426.686344,15.2800,258.720000,794.800,1004.09000,1451.2700
gross_sales,5000.0,3095.034344,2435.648960,25.5700,1128.840000,2386.180,4865.44000,9763.2500
discount_amount,5000.0,234.653715,285.742040,0.0000,1.462000,128.415,349.24425,1464.4875
net_sales,5000.0,2860.380629,2265.958863,21.7345,1043.154000,2156.120,4451.30550,9763.2500
total_cost,5000.0,2095.117696,1717.600926,15.2800,738.210000,1582.860,3191.76000,7256.3500
profit,5000.0,765.262933,711.175525,2.6845,248.907875,555.111,1066.50200,4372.6000
year,5000.0,2024.196600,0.682086,2023.0000,2024.000000,2024.000,2025.00000,2025.0000
